## 1. Import Library

In [1]:
import pandas as pd 
import numpy as np

## 2. Load Data Produsen

In [2]:
data_produsen = pd.read_csv('../data/raw/produsen/data_produsen.csv', parse_dates=['tanggal'])
data_produsen.head()

,tanggal,komoditas,titik_pantau,kabupaten,satuan,harga_kemarin,harga
0,2020-01-01,Beras,PS Bendul Mrisi,Kota Surabaya,kg,10200,10000
1,2020-01-01,Daging Sapi,RPH Pegirikan,Kota Surabaya,kg,91000,90000
2,2020-01-05,Beras,PS Bendul Mrisi,Kota Surabaya,kg,10000,10200
3,2020-01-05,Daging Sapi,RPH Pegirikan,Kota Surabaya,kg,90000,90000
4,2020-01-02,Beras,PS Bendul Mrisi,Kota Surabaya,kg,10000,10000


In [3]:
data_produsen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4872 entries, 0 to 4871
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   tanggal        4872 non-null   datetime64[ns]
 1   komoditas      4872 non-null   object        
 2   titik_pantau   4872 non-null   object        
 3   kabupaten      4872 non-null   object        
 4   satuan         4872 non-null   object        
 5   harga_kemarin  4872 non-null   int64         
 6   harga          4872 non-null   int64         
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 266.6+ KB


## 3. Preprocessing

Kalender kontinu (0 hari bolong), tidak perlu reindex grid. Missing tersamar sebagai 0 (13,4%).
Pelaporan berhenti: Beras valid terakhir 2026-02-16, Daging Sapi 2025-12-31 -> trim ekor (sekaligus buang gap 47/244 hari di ujung).
Gap tengah seri: 2x di 2020 (22 + 21 hari) -> sesuai aturan >14 hari tidak diisi, biarkan NaN + flag.
Tidak perlu drop komoditas. Urutan: trim -> 0 jadi NaN -> sort -> ffill/interpolasi per komoditas + flag imputed.

In [4]:
BATAS_VALID = {'Beras': '2026-02-16', 'Daging Sapi': '2025-12-31'}

for k, batas in BATAS_VALID.items():
    data_produsen = data_produsen[~((data_produsen['komoditas'] == k) & (data_produsen['tanggal'] > batas))]

data_produsen['harga'] = data_produsen['harga'].replace(0, np.nan)
data_produsen['harga_kemarin'] = data_produsen['harga_kemarin'].replace(0, np.nan)

data_produsen_clean = data_produsen.copy()
data_produsen_clean = data_produsen_clean.sort_values(['komoditas', 'tanggal']).reset_index(drop=True)
data_produsen_clean['imputed'] = data_produsen_clean['harga'].isna()
data_produsen_clean['harga'] = data_produsen_clean.groupby('komoditas')['harga'].transform(lambda s: s.ffill(limit=3).interpolate(limit=7))
data_produsen_clean['imputed'] = data_produsen_clean['imputed'] & data_produsen_clean['harga'].notna()

In [5]:
data_produsen_clean.head()

,tanggal,komoditas,titik_pantau,kabupaten,satuan,harga_kemarin,harga,imputed
0,2020-01-01,Beras,PS Bendul Mrisi,Kota Surabaya,kg,10200.0,10000.0,False
1,2020-01-02,Beras,PS Bendul Mrisi,Kota Surabaya,kg,10000.0,10000.0,False
2,2020-01-03,Beras,PS Bendul Mrisi,Kota Surabaya,kg,10000.0,10000.0,False
3,2020-01-04,Beras,PS Bendul Mrisi,Kota Surabaya,kg,10000.0,10000.0,False
4,2020-01-05,Beras,PS Bendul Mrisi,Kota Surabaya,kg,10000.0,10200.0,False


In [6]:
data_produsen_clean.info()
data_produsen_clean[data_produsen_clean['harga'].isna()].groupby('komoditas')['tanggal'].count()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4431 entries, 0 to 4430
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   tanggal        4431 non-null   datetime64[ns]
 1   komoditas      4431 non-null   object        
 2   titik_pantau   4431 non-null   object        
 3   kabupaten      4431 non-null   object        
 4   satuan         4431 non-null   object        
 5   harga_kemarin  4219 non-null   float64       
 6   harga          4351 non-null   float64       
 7   imputed        4431 non-null   bool          
dtypes: bool(1), datetime64[ns](1), float64(2), object(4)
memory usage: 246.8+ KB


komoditas
Beras          58
Daging Sapi    22
Name: tanggal, dtype: int64

## 4. Save ke Data Processed

In [7]:
data_produsen_clean.to_csv('../data/processed/data_produsen_clean.csv', index=False)
print(f"produsen: {len(data_produsen_clean):,} baris | {data_produsen_clean['komoditas'].nunique()} komoditas "
      f"| imputed {data_produsen_clean['imputed'].mean()*100:.1f}%")

produsen: 4,431 baris | 2 komoditas | imputed 3.0%
